# Query Expansion-based Retrieval Improvement in RAG


# Import Required Libraries

Initialize all dependencies required for vector search, embeddings, LLM interaction, and environment configuration.

In [40]:
import os
import chromadb

from dotenv import load_dotenv
from groq import Groq

from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Configure API Credentials

Load environment variables and initialize the Groq client for LLM inference.

In [41]:
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

client = Groq()

MODEL_NAME = "openai/gpt-oss-120b"

# Initialize Embedding Model

Load the sentence-transformer embedding model used for semantic retrieval.

In [42]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

# Connect to Persisted Vector Database

Load the local ChromaDB instance containing Tesla 10-K report embeddings.

In [43]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

COLLECTION_NAME = "tesla-10k-2019-to-2023"

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [44]:
chromadb_client

# Load Vector Store

Initialize the vector store and configure semantic similarity retrieval.

In [ ]:
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

TOP_K = 3

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


# Define Query Expansion Strategy

Create a prompt that generates multiple semantically equivalent versions of the user query.

In [12]:
query_expansion_system_message = """
You are an financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question \
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return 3 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template="""
<Question>
{question}
</Question>
"""

# User Query

Specify the financial question to be answered using retrieval-augmented generation.

In [13]:
user_query = "What was the automotive revenue in 2021?"


# Generate Alternative Query Variations

Use the LLM to create multiple phrasings of the original question for improved retrieval coverage.

In [45]:
prompt = [
    {
        "role": "system",
        "content": query_expansion_system_message
    },
    {
        "role": "user",
        "content": user_message_template.format(
            question=user_query
        )
    }
]

query_expansion = client.chat.completions.create(
    model=MODEL_NAME,
    messages=prompt,
    temperature=0.2
)

query_expansion_list = (
    query_expansion.choices[0]
    .message.content
    .strip()
    .split("\n")
)

query_expansion_list

['What was the automotive revenue in 2021?',
 'How much automotive revenue did the company generate in 2021?',
 'What were the automotive segment revenues for 2021?']

In [53]:
# Verify the Number of Queries Generated
for idx, query in enumerate(query_expansion_list, start=1):
    print(f"{idx}. {query}")

1. What was the automotive revenue in 2021?
2. How much automotive revenue did the company generate in 2021?
3. What were the automotive segment revenues for 2021?


# Retrieve Context Using Expanded Queries

Perform semantic search for each generated query variation and aggregate retrieved documents.

In [46]:
expanded_context_list = []

for query in query_expansion_list:
    expanded_context_list.extend(
        [doc.page_content for doc in retriever.invoke(query)]
    )

len(expanded_context_list)

9

In [48]:
expanded_context_list

['2022\n2021\n$\n%\n$\n%\nAutomotive\tsales\n$\n78,509\t\n$\n67,210\t\n$\n44,125\t\n$\n11,299\t\n17\t\n%\n$\n23,085\t\n52\t\n%\nAutomotive\tregulatory\tcredits\n1,790\t\n1,776\t\n1,465\t\n14\t\n1\t\n%\n311\t\n21\t\n%\nAutomotive\tleasing\n2,120\t\n2,476\t\n1,642\t\n(356)\n(14)\n%\n834\t\n51\t\n%\nTotal\tautomotive\trevenues\n82,419\t\n71,462\t\n47,232\t\n10,957\t\n15\t\n%\n24,230\t\n51\t\n%\nServices\tand\tother\n8,319\t\n6,091\t\n3,802\t\n2,228\t\n37\t\n%\n2,289\t\n60\t\n%\nTotal\tautomotive\t&\tservices\tand\tother\tsegment\nrevenue\n90,738\t\n77,553\t\n51,034\t\n13,185\t\n17\t\n%\n26,519\t\n52\t\n%\nEnergy\tgeneration\tand\tstorage\tsegment\trevenue\n6,035\t\n3,909\t\n2,789\t\n2,126\t\n54\t\n%\n1,120\t\n40\t\n%\nTotal\trevenues\n$\n96,773\t\n$\n81,462\t\n$\n53,823\t\n$\n15,311\t\n19\t\n%\n$\n27,639\t\n51\t\n%\nAutomotive\t&\tServices\tand\tOther\tSegment\nAutomotive\tsales\trevenue\tincludes\trevenues\trelated\tto\tcash\tand\tfinancing\tdeliveries\tof\tnew\tModel\tS,\tModel\tX,\tSem

# Deduplicate Retrieved Context

Remove duplicate document chunks retrieved from overlapping query variations.

In [47]:
final_context_documents = list(
    set(expanded_context_list)
)

len(final_context_documents)

4

In [ ]:
final_context_documents

['1,555\n\t\nGross\tprofit\n\t\n$\n18\n\t\n\t\n$\n190\n\t\n\t\n$\n190\n\t\n\t\nThe\tfollowing\ttable\tpresents\trevenues\tby\tgeographic\tarea\tbased\ton\tthe\tsales\tlocation\tof\tour\tproducts\t(in\tmillions):\n\t\n\t\n\t\nYear\tEnded\tDecember\t31,\n\t\n\t\n\t\n2020\n\t\n\t\n2019\n\t\n\t\n2018\n\t\nUnited\tStates\n\t\n$\n15,207\n\t\n\t\n$\n12,653\n\t\n\t\n$\n14,872\n\t\nChina\n\t\n\t\n6,662\n\t\n\t\n\t\n2,979\n\t\n\t\n\t\n1,757\n\t\nOther\n\t\n\t\n9,667\n\t\n\t\n\t\n8,946\n\t\n\t\n\t\n4,832\n\t\nTotal\n\t\n$\n31,536\n\t\n\t\n$\n24,578\n\t\n\t\n$\n21,461\n\t\n\t\nThe\trevenues\tin\tcertain\tgeographic\tareas\twere\timpacted\tby\tthe\tprice\tadjustments\twe\tmade\tto\tour\tvehicle\tofferings\tduring\tthe\tyears\tended\tDecember\n31,\t2020\tand\t2019.\tRefer\tto\tNote\t2,\t\nSummary\tof\tSignificant\tAccounting\tPolicies\n,\tfor\tdetails.\n\t\nThe\tfollowing\ttable\tpresents\tlong-lived\tassets\tby\tgeographic\tarea\t(in\tmillions):\n\t\n\t\n\t\nDecember\t31,\n\t\n\t\nDecember\t31,\n\t

In [ ]:
print("Expanded Queries:", len(query_expansion_list))
print("Retrieved Chunks:", len(expanded_context_list))
print("Unique Chunks:", len(final_context_documents))

Expanded Queries: 3
Retrieved Chunks: 9
Unique Chunks: 4


# Define Answer Generation Prompt

Create the final prompt that constrains the LLM to answer strictly from retrieved context.

In [50]:
final_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer of the question.

If the answer is not found in the context, respond "I don't know".
"""

final_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

# Generate Context-Aware Answer

Use the retrieved evidence to produce the final response grounded in the annual report data.

In [51]:
prompt = [
    {
        "role": "system",
        "content": final_system_message
    },
    {
        "role": "user",
        "content": final_user_message_template.format(
            context=final_context_documents,
            question=user_query
        )
    }
]

final_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=prompt,
    temperature=0
)

final_response.choices[0].message.content

'$47,232\u202fmillion.'